# 🦕 trikedb quickstart

**The single-file graph database.** A knowledge graph in one YAML file — full SPARQL 1.1, built for LLM agents.

This notebook builds a small knowledge graph, queries it with SPARQL, imports facts from a Markdown table, and renders the interactive graph **inline**.

In [ ]:
%pip install -q trikedb

## 1. Create a graph with an ontology

The ontology is the allowed vocabulary of predicates. Anything outside it is rejected — that is what keeps LLM-generated facts clean.

In [ ]:
from trikedb import TrikeDB, OntologyError

db = TrikeDB("pipeline.yaml", ontology={
    "PROVIDES":    "SaaS vendor -> ingestion job",
    "INGESTS_TO":  "ingestion job -> warehouse table",
    "MIGRATED_TO": "deprecated table -> its replacement",
    "AFFECTED_BY": "table -> change event",
})

db.add("salesflow-crm", "PROVIDES", "crm-sync-job")
db.add("crm-sync-job", "INGESTS_TO", "RAW_CRM_CONTACTS", schedule="hourly",
       via="https://api.salesflow.example/v2/contacts")
db.add("adastra-ads", "PROVIDES", "ads-spend-collector")
db.add("ads-spend-collector", "INGESTS_TO", "RAW_AD_SPEND_DAILY", schedule="daily 06:00")
db.add("LEGACY_CONTACTS", "MIGRATED_TO", "RAW_CRM_CONTACTS", deprecated=True)
db.add("RAW_AD_SPEND_DAILY", "AFFECTED_BY", "2025-04-01 adastra API v3: spend now in micros")

# node properties: type drives colors, label is the display name, the rest is free-form
db.set_node("salesflow-crm", type="saas", label="SalesFlow CRM", url="https://salesflow.example")
db.set_node("adastra-ads", type="saas")
db.set_node("crm-sync-job", type="job", owner="data-platform")
db.set_node("ads-spend-collector", type="job")
db.set_node("RAW_CRM_CONTACTS", type="table", pii=True)
db.set_node("RAW_AD_SPEND_DAILY", type="table")

db.save()
db

The ontology guard in action — a hallucinated predicate is rejected:

In [ ]:
try:
    db.add("crm-sync-job", "TOTALLY_MADE_UP", "x")
except OntologyError as e:
    print("rejected:", e)

## 2. Query it

Zero-dependency pattern matching, or real SPARQL 1.1 (rdflib engine). Writes go through SPARQL too and land back in the YAML.

In [ ]:
db.query(["?vendor PROVIDES ?job", "?job INGESTS_TO ?table"])

In [ ]:
db.sparql("""
  SELECT ?vendor ?table WHERE {
    ?vendor t:PROVIDES ?job .
    ?job t:INGESTS_TO ?table .
  }
""")

In [ ]:
db.sparql("INSERT DATA { t:figly t:PROVIDES t:figly-export-job }")
db.save()
print(db.sparql("ASK { t:figly t:PROVIDES ?j }"))

## 3. Import from a Markdown doc

Any table whose header has s/p/o columns is picked up; prose and other tables are ignored. Your design docs are data.

In [ ]:
open("design_note.md", "w").write("""
# Q3 ingestion changes (an ordinary design doc)

| s              | p          | o                  | schedule  |
|----------------|------------|--------------------|-----------|
| clickpath-pa   | PROVIDES   | clickpath-webhook  |           |
| clickpath-webhook | INGESTS_TO | RAW_PRODUCT_EVENTS | streaming |
""")
print("imported:", db.import_file("design_note.md"), "triples")
db.save()
db

## 4. See the graph — right here in the notebook

The HTML export is a full workbench (click nodes for details, search, a SPARQL console). Embedding it via `srcdoc` makes it work inside the notebook output.

In [ ]:
import html as html_mod
from IPython.display import HTML

page = db.to_html(title="pipeline.yaml")
HTML(f'<iframe srcdoc="{html_mod.escape(page)}" width="100%" height="620" style="border:1px solid #ddd; border-radius:8px;"></iframe>')

## 5. The database is just a file

Everything above lives in `pipeline.yaml` — read it, diff it, commit it, hand it to an agent.

In [ ]:
print(open("pipeline.yaml").read())

## Next steps

- Keep `graph.yaml` in your repo and tell your agent to read it before data work
- Or serve it as an ontology layer over MCP: `trikedb mcp graph.yaml` (stdio, for local agent sessions)
- Docs & source: https://github.com/RyutoYoda/trikedb